In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

c:\Users\richa\Downloads\CS4248\FinalProject\CS4248_Project


In [2]:
import json
import pandas as pd

# load anchored cleaned data
df = pd.read_json("../artifacts/data/cleaned_with_anchors.jsonl", lines=True)

# load split ids
with open("../artifacts/splits/standard.json", "r", encoding="utf-8") as f:
    split = json.load(f)

dev_ids = set(split["dev"])
dev_df = df[df["id"].isin(dev_ids)].copy()

print(len(dev_df))
print(dev_df.columns)
print(dev_df[["id", "text", "label"]].head())

2850
Index(['id', 'headline_raw', 'text', 'label', 'article_link', 'publisher',
       'anchors'],
      dtype='str')
            id                                               text  label
2   sar_000003  eat your veggies: 9 deliciously different recipes      0
10  sar_000011  this lesbian is considered a father in indiana...      0
14  sar_000015  ford develops new suv that runs purely on gaso...      1
19  sar_000020  lin-manuel miranda would like to remind you to...      0
21  sar_000022  guard in video game under strict orders to rep...      1


In [3]:
dev_df[dev_df["label"] == 1][["id", "text", "anchors"]].head(10)

,id,text,anchors
14,sar_000015,ford develops new suv that runs purely on gaso...,"{'entities': [{'text': 'ford', 'start': 0, 'en..."
21,sar_000022,guard in video game under strict orders to rep...,"{'entities': [], 'numbers': [], 'capitals': []..."
100,sar_000101,report: 70% of trump endorsements made after s...,"{'entities': [], 'numbers': [{'text': '70%', '..."
104,sar_000105,dad immediately hands phone to mom,"{'entities': [], 'numbers': [], 'capitals': []..."
143,sar_000144,stomach sets aside synthetic additives until i...,"{'entities': [], 'numbers': [], 'capitals': []..."
147,sar_000148,south dakota asked to water north dakota's cro...,"{'entities': [{'text': 'south dakota', 'start'..."
183,sar_000184,huckabee sanders tells colleagues she's taking...,"{'entities': [{'text': 'google', 'start': 65, ..."
195,sar_000196,new anti-drug program teaches teens to resist ...,"{'entities': [], 'numbers': [], 'capitals': []..."
198,sar_000199,there no way tv character could actually affor...,"{'entities': [{'text': 'new york city'', 'star..."
224,sar_000225,complete idiot forgot to shave area between mo...,"{'entities': [], 'numbers': [], 'capitals': []..."


In [4]:
dev_df[dev_df["label"] == 0][["id", "text", "anchors"]].head(10)

,id,text,anchors
2,sar_000003,eat your veggies: 9 deliciously different recipes,"{'entities': [], 'numbers': [{'text': '9', 'st..."
10,sar_000011,this lesbian is considered a father in indiana...,"{'entities': [{'text': 'indiana', 'start': 39,..."
19,sar_000020,lin-manuel miranda would like to remind you to...,"{'entities': [{'text': 'lin-manuel miranda', '..."
22,sar_000023,how to live to be 110,"{'entities': [], 'numbers': [{'text': '110', '..."
26,sar_000027,this new orange era: the growing divide,"{'entities': [], 'numbers': [], 'capitals': []..."
89,sar_000090,exclusive promo hints stephen colbert will unl...,"{'entities': [], 'numbers': [], 'capitals': []..."
98,sar_000099,nick cannon responds to mariah carey's engagem...,"{'entities': [{'text': 'nick cannon', 'start':..."
105,sar_000106,"my worst audition ever? or, the danger of play...","{'entities': [], 'numbers': [], 'capitals': []..."
123,sar_000124,rnc leader to trump: tone it down!,"{'entities': [], 'numbers': [], 'capitals': []..."
132,sar_000133,rick perry on donald trump's proposed border w...,"{'entities': [{'text': 'rick perry', 'start': ..."


In [5]:
dev_df.iloc[0]["anchors"]

{'entities': [],
 'numbers': [{'text': '9', 'start': 18, 'end': 19, 'type': 'number'}],
 'capitals': [],
 'all': [{'text': '9', 'start': 18, 'end': 19, 'type': 'number'}]}

In [6]:
def get_anchor_texts(anchor_dict):
    return [a["text"] for a in anchor_dict.get("all", [])]

def preserves_anchors(anchor_dict, rewritten_text):
    anchor_texts = get_anchor_texts(anchor_dict)
    rewritten_lower = rewritten_text.lower()
    return all(anchor.lower() in rewritten_lower for anchor in anchor_texts)

def neutral_to_sarcastic_v1(text):
    return "Report: " + text

sample = dev_df[dev_df["label"] == 0].head(5)

for _, row in sample.iterrows():
    rewritten = neutral_to_sarcastic_v1(row["text"])
    ok = preserves_anchors(row["anchors"], rewritten)
    print("ORIG:", row["text"])
    print("NEW :", rewritten)
    print("ANCHORS:", get_anchor_texts(row["anchors"]))
    print("PRESERVED:", ok)
    print("-" * 60)

ORIG: eat your veggies: 9 deliciously different recipes
NEW : Report: eat your veggies: 9 deliciously different recipes
ANCHORS: ['9']
PRESERVED: True
------------------------------------------------------------
ORIG: this lesbian is considered a father in indiana (and an amazing one at that)
NEW : Report: this lesbian is considered a father in indiana (and an amazing one at that)
ANCHORS: ['indiana']
PRESERVED: True
------------------------------------------------------------
ORIG: lin-manuel miranda would like to remind you to put your phone away
NEW : Report: lin-manuel miranda would like to remind you to put your phone away
ANCHORS: ['lin-manuel miranda']
PRESERVED: True
------------------------------------------------------------
ORIG: how to live to be 110
NEW : Report: how to live to be 110
ANCHORS: ['110']
PRESERVED: True
------------------------------------------------------------
ORIG: this new orange era: the growing divide
NEW : Report: this new orange era: the growing divi

In [7]:
from systems.system_a.system_a_template import SystemATemplate

system_a = SystemATemplate()

sample = dev_df[dev_df["label"] == 0].head(5)

for _, row in sample.iterrows():
    rewritten = system_a.rewrite_row(row, direction="n2s")
    print("ORIG:", row["text"])
    print("NEW :", rewritten)
    print("ANCHORS:", row["anchors"]["all"])
    print("-" * 60)

ORIG: eat your veggies: 9 deliciously different recipes
NEW : eat your veggies: 9 deliciously different recipes in stunning development
ANCHORS: [{'text': '9', 'start': 18, 'end': 19, 'type': 'number'}]
------------------------------------------------------------
ORIG: this lesbian is considered a father in indiana (and an amazing one at that)
NEW : Experts confirm this lesbian is considered a father in indiana (and an amazing one at that)
ANCHORS: [{'text': 'indiana', 'start': 39, 'end': 46, 'type': 'entity'}]
------------------------------------------------------------
ORIG: lin-manuel miranda would like to remind you to put your phone away
NEW : Experts confirm lin-manuel miranda would like to remind you to put your phone away
ANCHORS: [{'text': 'lin-manuel miranda', 'start': 0, 'end': 18, 'type': 'entity'}]
------------------------------------------------------------
ORIG: how to live to be 110
NEW : how to live to be 110 for some reason
ANCHORS: [{'text': '110', 'start': 18, 'end'

In [8]:
row = dev_df[dev_df["label"] == 0].iloc[0]
system_a.get_candidates(row["text"], row["anchors"], "n2s")

['eat your veggies: 9 deliciously different recipes in stunning development',
 "eat your veggies: 9 deliciously different recipes to everyone's surprise",
 'Experts confirm eat your veggies: 9 deliciously different recipes',
 'Nation relieved as eat your veggies: 9 deliciously different recipes',
 'Report: eat your veggies: 9 deliciously different recipes',
 'Sources confirm eat your veggies: 9 deliciously different recipes']

In [9]:
sample = dev_df[dev_df["label"] == 1].head(5)

for _, row in sample.iterrows():
    rewritten = system_a.rewrite_row(row, direction="s2n")
    print("ORIG:", row["text"])
    print("NEW :", rewritten)
    print("ANCHORS:", row["anchors"]["all"])
    print("-" * 60)

ORIG: ford develops new suv that runs purely on gasoline
NEW : ford develops new suv that runs purely on gasoline
ANCHORS: [{'text': 'ford', 'start': 0, 'end': 4, 'type': 'entity'}]
------------------------------------------------------------
ORIG: guard in video game under strict orders to repeatedly pace same stretch of hallway
NEW : guard in video game under strict orders to repeatedly pace same stretch of hallway
ANCHORS: []
------------------------------------------------------------
ORIG: report: 70% of trump endorsements made after staring at bedroom ceiling for 4 hours
NEW : 70% of trump endorsements made after staring at bedroom ceiling for 4 hours
ANCHORS: [{'text': '70%', 'start': 8, 'end': 11, 'type': 'number'}, {'text': '4', 'start': 76, 'end': 77, 'type': 'number'}]
------------------------------------------------------------
ORIG: dad immediately hands phone to mom
NEW : dad immediately hands phone to mom
ANCHORS: []
------------------------------------------------------

In [10]:
from systems.system_a.system_a_template import SystemATemplate
from systems.system_a.template_utils import preserves_anchors

system_a = SystemATemplate()

sample = dev_df.head(20)

for _, row in sample.iterrows():
    direction = "n2s" if row["label"] == 0 else "s2n"
    rewritten = system_a.rewrite_row(row, direction=direction)

    print("LABEL:", row["label"])
    print("ORIG :", row["text"])
    print("NEW  :", rewritten)
    print("ANCHORS:", row["anchors"]["all"])
    print("PRESERVED:", preserves_anchors(row["anchors"], rewritten))
    print("-" * 80)

LABEL: 0
ORIG : eat your veggies: 9 deliciously different recipes
NEW  : eat your veggies: 9 deliciously different recipes in stunning development
ANCHORS: [{'text': '9', 'start': 18, 'end': 19, 'type': 'number'}]
PRESERVED: True
--------------------------------------------------------------------------------
LABEL: 0
ORIG : this lesbian is considered a father in indiana (and an amazing one at that)
NEW  : Experts confirm this lesbian is considered a father in indiana (and an amazing one at that)
ANCHORS: [{'text': 'indiana', 'start': 39, 'end': 46, 'type': 'entity'}]
PRESERVED: True
--------------------------------------------------------------------------------
LABEL: 1
ORIG : ford develops new suv that runs purely on gasoline
NEW  : ford develops new suv that runs purely on gasoline
ANCHORS: [{'text': 'ford', 'start': 0, 'end': 4, 'type': 'entity'}]
PRESERVED: True
--------------------------------------------------------------------------------
LABEL: 0
ORIG : lin-manuel miranda wou

In [11]:
row = dev_df[dev_df["label"] == 0].iloc[0]
system_a.get_candidates(row["text"], row["anchors"], "n2s")

['eat your veggies: 9 deliciously different recipes in stunning development',
 "eat your veggies: 9 deliciously different recipes to everyone's surprise",
 'Experts confirm eat your veggies: 9 deliciously different recipes',
 'Nation relieved as eat your veggies: 9 deliciously different recipes',
 'Report: eat your veggies: 9 deliciously different recipes',
 'Sources confirm eat your veggies: 9 deliciously different recipes']

In [12]:
sample = dev_df.head(20)

for _, row in sample.iterrows():
    direction = "n2s" if row["label"] == 0 else "s2n"
    rewritten = system_a.rewrite_row(row, direction=direction)

    print("LABEL:", row["label"])
    print("ORIG :", row["text"])
    print("NEW  :", rewritten)
    print("ANCHORS:", row["anchors"]["all"])
    print("-" * 80)

LABEL: 0
ORIG : eat your veggies: 9 deliciously different recipes
NEW  : eat your veggies: 9 deliciously different recipes in stunning development
ANCHORS: [{'text': '9', 'start': 18, 'end': 19, 'type': 'number'}]
--------------------------------------------------------------------------------
LABEL: 0
ORIG : this lesbian is considered a father in indiana (and an amazing one at that)
NEW  : Experts confirm this lesbian is considered a father in indiana (and an amazing one at that)
ANCHORS: [{'text': 'indiana', 'start': 39, 'end': 46, 'type': 'entity'}]
--------------------------------------------------------------------------------
LABEL: 1
ORIG : ford develops new suv that runs purely on gasoline
NEW  : ford develops new suv that runs purely on gasoline
ANCHORS: [{'text': 'ford', 'start': 0, 'end': 4, 'type': 'entity'}]
--------------------------------------------------------------------------------
LABEL: 0
ORIG : lin-manuel miranda would like to remind you to put your phone away
NEW

In [13]:
import random

sample0 = dev_df[dev_df["label"] == 0].sample(50, random_state=42)
sample1 = dev_df[dev_df["label"] == 1].sample(50, random_state=42)
sample = pd.concat([sample0, sample1])

for _, row in sample.head(20).iterrows():
    direction = "n2s" if row["label"] == 0 else "s2n"
    rewritten = system_a.rewrite_row(row, direction=direction)
    print("LABEL:", row["label"])
    print("ORIG :", row["text"])
    print("NEW  :", rewritten)
    print("-" * 80)

LABEL: 0
ORIG : georgia executes gregory lawler for killing police officer, despite autism defense
NEW  : Experts confirm georgia executes gregory lawler for killing police officer, despite autism defense
--------------------------------------------------------------------------------
LABEL: 0
ORIG : senate republicans just killed their health care bill again
NEW  : Nation relieved as senate republicans just killed their health care bill again
--------------------------------------------------------------------------------
LABEL: 0
ORIG : 10 things no one told me before my c-section
NEW  : 10 things no one told me before my c-section in stunning development
--------------------------------------------------------------------------------
LABEL: 0
ORIG : letter to my girls about the mean girl
NEW  : Experts confirm letter to my girls about the mean girl
--------------------------------------------------------------------------------
LABEL: 0
ORIG : south korea on heightened alert as nort